**1: Налаштування середовища (Setup)**

In [1]:
%%capture
import torch
import os
major_version, minor_version = torch.cuda.get_device_capability()

# Встановлюємо Unsloth та залежності
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

os.environ["WANDB_DISABLED"] = "true"

**2: Завантаження даних та ініціалізація моделі**

In [2]:
from google.colab import files
from unsloth import FastLanguageModel

print("Будь ласка, завантаж підготовлений датасет train.json")
uploaded = files.upload()
filename = next(iter(uploaded))

max_seq_length = 2048
# Завантажуємо базову квантовану модель
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Застосовуємо LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)
print("Модель успішно завантажена та готова до донавчання!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Будь ласка, завантаж підготовлений датасет train.json


Saving train.json to train (1).json
==((====))==  Unsloth 2026.3.15: Fast Mistral patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.3.15 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Модель успішно завантажена та готова до донавчання!


**3: Процес Fine-Tuning**

In [3]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = load_dataset("json", data_files = {"train": filename}, split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)
print("Починаю навчання...")
trainer_stats = trainer.train()

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

Починаю навчання...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Step,Training Loss
1,1.158935
2,1.086322
3,1.033338
4,0.846609
5,0.860177
6,0.755776
7,0.842912
8,0.709926
9,0.656206
10,0.664019


**4: Тест інференсу (Генерація саммарі)**

In [5]:
# Перемикаємось у режим генерації
FastLanguageModel.for_inference(model)

text_to_summarize = """
КАБІНЕТ МІНІСТРІВ УКРАЇНИ
ПОСТАНОВА від 27 травня 2022 р. № 634
Про особливості оренди державного та комунального майна у період воєнного стану.
Кабінет Міністрів України постановляє:
1. Установити, що на період воєнного стану:
1) орендарі звільняються від орендної плати за договорами оренди державного та комунального майна, розміщеного на територіях, де ведуться активні бойові дії;
2) договори оренди, строк дії яких завершується у період воєнного стану, вважаються автоматично продовженими на період дії воєнного стану та чотири місяці після його припинення чи скасування;
"""

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Створи стисле юридичне резюме (summary) для наступного документу.",
        text_to_summarize,
        "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = False)
result = tokenizer.batch_decode(outputs)[0]

print("\n=== РЕЗУЛЬТАТ МОДЕЛІ: ===")
final_text = result.split("### Response:")[1].replace("<|endoftext|>", "").replace("</s>", "").strip()
print(final_text)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12


=== РЕЗУЛЬТАТ МОДЕЛІ: ===
Про особливості оренди державного та комунального майна у період воєнного стану.


**5: Експорт у формат GGUF**

In [6]:
from google.colab import drive

drive.mount('/content/drive')
print("Починаю конвертацію моделі в GGUF...")

# Зберігаємо квантовану модель
model.save_pretrained_gguf("Legal-Mistral-7B-v0.3", tokenizer, quantization_method = "q4_k_m")

# Копіюємо на диск
!mkdir -p "/content/drive/My Drive/Legal_Diploma"
!cp "/content/Legal-Mistral-7B-v0.3_gguf/mistral-7b-instruct-v0.3.Q4_K_M.gguf" "/content/drive/My Drive/Legal_Diploma/"

print("ГОТОВО! Файл .gguf успішно збережено на Google Drive.")

Mounted at /content/drive
Починаю конвертацію моделі в GGUF...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00003.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

Downloaded tokenizer.model



Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [08:23<00:00, 167.72s/it]


Unsloth: Merge process complete. Saved to `/content/Legal-Mistral-7B-v0.3`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['Legal-Mistral-7B-v0.3_gguf/mistral-7b-instruct-v0.3.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['Legal-Mistral-7B-v0.3_gguf/mistral-7b-instruct-v0.3.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model Legal-Mistral-7B-v0.3_gguf/mistral-7b-instruct-v0.3.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to Legal-Mistral-7B-v0.3_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f Legal-Mistral-7B-v0.3_gguf/Modelfile
ГОТОВО! Файл .gguf успішно збережено на Google Drive.
